In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import itertools
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap
# Optional: higher numeric accuracy (do this before heavy JAX use)
try:
    from jax import config as jax_config
    jax_config.update("jax_enable_x64", True)
except Exception:
    pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('default')
plt.close('all')
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
mpl.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
key = jax.random.key(1701)

x = jnp.linspace(0, 10, 500)
y = jnp.sin(x) + 0.2 * jax.random.normal(key, shape=(500,))

window = jnp.ones(10) / 10
y_smooth = jnp.convolve(y, window, mode='same')

plt.plot(x, y, 'lightgray')
plt.plot(x, y_smooth, 'black');

In [ ]:
lya = sjx.constants.lines.get_line("LyA")
print(f"Lyman-alpha λ₀ = {lya.lambda0_A} Å, f = {lya.f_osc}")

In [ ]:
Line_profiles = {}
xx = jnp.linspace(-20, 20, 2**13)     # for reference (NumPy/SciPy)

σ_low = 1e-3
σ_high = 1.0
γ_low = 1e-3
γ_high = 1.0

params = jnp.array([ # (σ, γ) pairs
    (σ_low, γ_low),
    (σ_high, γ_low),
    (σ_low, γ_high),
    (σ_high, γ_high),
])

Line_profiles["Voigt"] = {}
Line_profiles["Voigt"]['Faddeva'] = sjx.optical_depth.line_profile.Voigt_np(xx, params[:, 0], params[:, 1])
Line_profiles["Voigt"]['JAX_conv'] = sjx.optical_depth.line_profile.Voigt(xx, params[:, 0], params[:, 1])
Line_profiles["Voigt"]['JAX_TG'] = sjx.optical_depth.line_profile.Voigt_TG_profile(xx, params[:, 0], params[:, 1])

Line_profiles["Lorentzian"] = sjx.optical_depth.line_profile.Lorentzian(xx, jnp.array([γ_low, γ_high]))
Line_profiles["Gaussian"] = sjx.optical_depth.line_profile.Gaussian(xx, jnp.array([σ_low, σ_high]))

In [ ]:
V_fad = np.asarray(Line_profiles["Faddeva"])   # (P, N)
V_conv = np.asarray(Line_profiles["JAX_conv"]) # (P, N)
V_tg   = np.asarray(Line_profiles["JAX_TG"])   # (P, N)

P = len(params)
colors = np.array(plt.cm.get_cmap("tab10").colors[:P])

# Two-row layout: top main (2/3), bottom residuals (1/3), shared x-axis
fig, (ax, ax_res) = plt.subplots(
    2, 1, figsize=(9, 7), sharex=True,
    gridspec_kw=dict(height_ratios=[2, 1])
)

for ii in range(P):
    sigma, gamma = params[ii]
    color = colors[ii]

    ax.plot(xx, V_fad[ii], color=color, linestyle="-", linewidth=1)
    ax.plot(xx, V_conv[ii], color=color, linestyle="--", linewidth=3)
    # ax.plot(xx, V_tg[ii],   color=color, linestyle=":",  linewidth=6)

method_handles = [
    Line2D([0], [0], color="k", lw=8,  label="Scipy Faddeeva"),
    Line2D([0], [0], color="k", lw=4,  ls="--", label="JAX convolution"),
    Line2D([0], [0], color="k", lw=2,  ls=":",  label="JAX TG approx."),
]
leg1 = ax.legend(handles=method_handles, loc="upper right", fontsize=16)
ax.add_artist(leg1)

param_handles = []
for ii in range(P):
    sigma, gamma = params[ii]
    lbl = rf"$\sigma={sigma:g},\ \gamma={gamma:g}$"
    param_handles.append(Line2D([0], [0], color=colors[ii], lw=6, label=lbl))

leg2 = ax.legend(handles=param_handles, loc="upper left", fontsize=16)
ax.add_artist(leg2)

ax.set_title("Voigt profiles")
# ax.set_xlabel("x")   # <-- removed (top panel has no x-label)
ax.set_ylabel(r"$V(x;\ \sigma,\ \gamma)$")
ax.set_yscale("log")

# ====== Residuals subpanel: model − reference ======
for ii in range(P):
    color = colors[ii]
    ax_res.plot(xx, V_conv[ii] / V_fad[ii] - 1, color=color, linestyle="--", linewidth=2)
    # ax_res.plot(xx, V_tg[ii] - V_fad[ii], color=color, linestyle=":", linewidth=1)

ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res.set_xlabel("x")        # only bottom panel gets the x-label
ax_res.set_ylabel("residual")
# ax_res.set_yscale("symlog", linthresh=1e-6)

plt.subplots_adjust(hspace=0.05)
fig.tight_layout()
plt.show()